# **Barkod Oluşturma ve Okuma**

- Bu derste çeşitli standartlarda barkodlar oluşturacak ve üzerlerinde ne olduğunu okuyacağız.

In [ ]:
# Kurulumumuz
!pip install python-barcode[images]
!pip install qrcode
!apt install libzbar0
!pip install pyzbar

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt


def imshow(title = "Image", image = None, size = 10):
    w, h = image.shape[0], image.shape[1]
    aspect_ratio = w/h
    plt.figure(figsize=(size * aspect_ratio,size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.show()

## **Barkod Üretimi** 
Python-barcode paketimizi kullanarak barkod üretelim.

Desteklenen Formatlar
Bu yazının yazıldığı sırada, bu paket aşağıdaki formatları desteklemektedir:
- EAN-8
- EAN-13
- EAN-14
- UPC-A
- JAN
- ISBN-10
- ISBN-13
- ISSN
- Code 39
- Code 128
- PZN


In [ ]:
from barcode import EAN13
from barcode.writer import ImageWriter

with open('barcode.png', 'wb') as f:
    EAN13('123456789102', writer=ImageWriter()).write(f)

barcode = cv2.imread("barcode.png")
imshow("Barcode", barcode)

## **QR Kod Üretimi** 
qrcode paketimizi kullanarak QR Kodları oluşturalım.

QR kodu (Quick Response code'dan kısaltılmıştır) ilk olarak 1994 yılında Japonya'da otomotiv endüstrisi için tasarlanmış bir matris barkod (veya iki boyutlu barkod) türüdür. Barkod, üzerine iliştirildiği öğe hakkında bilgi içeren, makine tarafından okunabilen optik bir etikettir. Uygulamada, QR kodları genellikle bir web sitesine veya uygulamaya işaret eden bir konum belirleyici, tanımlayıcı veya izleyici için veri içerir. Bir QR kodu, verileri verimli bir şekilde depolamak için dört standartlaştırılmış kodlama modu (sayısal, alfanümerik, bayt/ikili ve kanji) kullanır; uzantılar da kullanılabilir.

Bir QR kodu, beyaz bir arka plan üzerinde kare bir ızgara şeklinde düzenlenmiş siyah karelerden oluşur; bu kareler kamera gibi bir görüntüleme cihazı tarafından okunabilir ve görüntü uygun şekilde yorumlanana kadar Reed-Solomon hata düzeltme kullanılarak işlenebilir. Gerekli veriler daha sonra görüntünün hem yatay hem de dikey bileşenlerinde bulunan desenlerden çıkarılır.

<img src="QR_Code_Structure_Example_3.jpg" width="500">

In [ ]:
import qrcode
from PIL import Image

qr = qrcode.QRCode(
    version=1,
    error_correction=qrcode.constants.ERROR_CORRECT_H,
    box_size=10,
    border=4,
)

qr.add_data("https://wwww.opencv.org")
qr.make(fit=True)
img = qr.make_image(fill_color="black", back_color="white")
img.save("qrcode.png")

qrcode = cv2.imread("qrcode.png")
imshow("QR Code", qrcode, size = 8)

**QR Kodları için Yapılandırma**:

- version — QR Kodunun boyutunu kontrol eder. 1'den 40'a kadar bir tamsayı kabul eder. Sürüm 1 21 x 21 matristen oluşur.
- error_correction — QR Kodu için kullanılan hata düzeltmeyi kontrolü.
- box_size — QR kodunun her kutusunun piksel sayısının kontrolü.
- border — Kenarlığın kutu kalınlığının kontrolü. Varsayılan değer 4'tür ve bu da spesifikasyona göre minimum değerdir.

error_correction için 4 sabit mevcuttur. Hatalar ne kadar yüksek oranda düzeltilebilirse o kadar iyidir.

- ERROR_CORRECT_L — Yaklaşık %7 veya daha az hata düzeltilebilir.
- ERROR_CORRECT_M — Yaklaşık %15 veya daha az hata düzeltilebilir. Bu varsayılan değerdir.
- ERROR_CORRECT_Q — Yaklaşık %25 veya daha az hata düzeltilebilir.
- ERROR_CORRECT_H — Yaklaşık %30 veya daha az hata düzeltilebilir.

## **Karekodları Çözme**

In [ ]:
from pyzbar.pyzbar import decode
from PIL import Image

img = Image.open('qrcode.png')
result = decode(img)
for i in result:
    print(i.data.decode("utf-8"))

### **Karekodları (QR kodları) Algılama**

In [ ]:
from pyzbar.pyzbar import decode

image = cv2.imread("../files/images/1DwED.jpg")

# qrcode'u algılayın ve kodunu çözün
codes = decode(image)

# tespit edilen barkodlar üzerinde döngü
for bc in codes:
  # Metin yerleşimi için dikdörtgen koordinatlarını alalım
  (x, y, w, h) = bc.rect
  print(bc.polygon)
  pt1,pt2,pt3,pt4 = bc.polygon

  # Algılanan QR kodu üzerine bir sınırlayıcı kutu çizin
  pts = np.array( [[pt1.x,pt1.y], [pt2.x,pt2.y], [pt3.x,pt3.y], [pt4.x,pt4.y]], np.int32)
  pts = pts.reshape((-1,1,2))
  cv2.polylines(image, [pts], True, (0,0,255), 3)

  # Nesnemizden dize bilgi verilerini ve türünü çıkaralım
  barcode_text = bc.data.decode()
  barcode_type = bc.type

  # göster 
  text = "{} ({})".format(barcode_text, barcode_type)
  cv2.putText(image, barcode_text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 3)
  cv2.putText(image, barcode_type, (x+w, y+h - 15), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 3)
  print("QR Code revealed: {}".format(text))

# çıktımızı görüntüleyelim
imshow("QR Scanner", image, size = 12)

In [ ]:
from pyzbar.pyzbar import decode

image = cv2.imread("../files/images/1024px-ISBN.jpg")

# qrcode'u algılayın ve kodunu çözün
barcodes = decode(image)

# tespit edilen barkodlar üzerinde döngü
for bc in barcodes:
  # Metin yerleşimimiz için dikdörtgen koordinatlarını alalım
  (x, y, w, h) = bc.rect
  cv2.rectangle(image, (x, y), (x + w, y + h), (255, 0, 0), 3)

  # Nesnemizden dize bilgi verilerini ve türünü çıkaralım
  barcode_text = bc.data.decode()
  barcode_type = bc.type

  # göster
  text = "{} ({})".format(barcode_text, barcode_type)
  cv2.putText(image, barcode_text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 3)
  cv2.putText(image, barcode_type, (x+w, y+h - 15), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 3)
  print("Barcode revealed: {}".format(barcode_text))
  print("Barcode revealed: {}".format(barcode_text))

# çıktımızı görüntüleyelim
imshow("QR Scanner", image, size = 16)